In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 3, Finished, Available, Finished, False)

In [2]:
landing_path = "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing"
df_trailers = (
    spark.read
    .option("header", "true")
    .csv(landing_path + "/trailers.csv")
)

display(df_trailers)

print(f"Source records: {df_trailers.count()}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d51e8b5d-0245-472b-8e82-4bd47437c9f7)

Source records: 180


In [3]:
df_trailers.printSchema()

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 5, Finished, Available, Finished, False)

root
 |-- trailer_id: string (nullable = true)
 |-- trailer_number: string (nullable = true)
 |-- trailer_type: string (nullable = true)
 |-- length_feet: string (nullable = true)
 |-- model_year: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- current_location: string (nullable = true)



In [4]:
print(df_trailers.columns)

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 6, Finished, Available, Finished, False)

['trailer_id', 'trailer_number', 'trailer_type', 'length_feet', 'model_year', 'vin', 'acquisition_date', 'status', 'current_location']


In [5]:
trailer_schema = StructType([
    StructField("trailer_id", StringType(), True),
    StructField("trailer_number", StringType(), True),
    StructField("trailer_type", StringType(), True),
    StructField("length_feet", DoubleType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("vin", StringType(), True),
    StructField("acquisition_date", DateType(), True),
    StructField("status", StringType(), True),
    StructField("current_location", StringType(), True)
])

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 7, Finished, Available, Finished, False)

In [6]:
df_trailers = (
    spark.read
    .option("header", "true")
    .schema(trailer_schema)
    .csv(landing_path + "/trailers.csv")
)

display(df_trailers)

print(f"Source records: {df_trailers.count()}")

df_trailers.printSchema()

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a5a11d4-5961-488c-adda-fea6dc7d6444)

Source records: 180
root
 |-- trailer_id: string (nullable = true)
 |-- trailer_number: string (nullable = true)
 |-- trailer_type: string (nullable = true)
 |-- length_feet: double (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- current_location: string (nullable = true)



In [7]:
source_count = df_trailers.count()
print(f"Source records: {source_count}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 9, Finished, Available, Finished, False)

Source records: 180


In [8]:
null_trailer_ids = (
    df_trailers
    .filter(F.col("trailer_id").isNull())
    .count()
)

print(f"NULL trailer IDs: {null_trailer_ids}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 10, Finished, Available, Finished, False)

NULL trailer IDs: 0


In [9]:
duplicate_trailer_ids = (
    df_trailers
    .groupBy("trailer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate trailer IDs: {duplicate_trailer_ids}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 11, Finished, Available, Finished, False)

Duplicate trailer IDs: 0


In [10]:
invalid_length = (
    df_trailers
    .filter(F.col("length_feet") <= 0)
    .count()
)
print(f"Invalid trailer length: {invalid_length}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 12, Finished, Available, Finished, False)

Invalid trailer length: 0


In [11]:
invalid_model_year = (
    df_trailers
    .filter(
        (F.col("model_year") < 1900) |
        (F.col("model_year") > 2100)
    )
    .count()
)

print(f"Invalid model years: {invalid_model_year}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 13, Finished, Available, Finished, False)

Invalid model years: 0


In [12]:
null_acquisition_dates = (
    df_trailers
    .filter(F.col("acquisition_date").isNull())
    .count()
)

print(f"NULL acquisition dates: {null_acquisition_dates}")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 14, Finished, Available, Finished, False)

NULL acquisition dates: 0


In [13]:
df_trailers = (
    df_trailers
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("trailers.csv"))
)

df_trailers.createOrReplaceTempView("trailers_source")

print("Trailer source data prepared for Bronze layer.")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 15, Finished, Available, Finished, False)

Trailer source data prepared for Bronze layer.


In [14]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_trailers (
    trailer_id STRING,
    trailer_number STRING,
    trailer_type STRING,
    length_feet DOUBLE,
    model_year INT,
    vin STRING,
    acquisition_date DATE,
    status STRING,
    current_location STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_trailers table is ready.")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 16, Finished, Available, Finished, False)

bronze_trailers table is ready.


In [15]:
result = spark.sql("""
MERGE INTO bronze_trailers AS target
USING trailers_source AS source

ON target.trailer_id = source.trailer_id

WHEN MATCHED THEN
    UPDATE SET
        target.trailer_number = source.trailer_number,
        target.trailer_type = source.trailer_type,
        target.length_feet = source.length_feet,
        target.model_year = source.model_year,
        target.vin = source.vin,
        target.acquisition_date = source.acquisition_date,
        target.status = source.status,
        target.current_location = source.current_location,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        trailer_id,
        trailer_number,
        trailer_type,
        length_feet,
        model_year,
        vin,
        acquisition_date,
        status,
        current_location,
        ingestion_timestamp,
        source_file
    )
    VALUES (
        source.trailer_id,
        source.trailer_number,
        source.trailer_type,
        source.length_feet,
        source.model_year,
        source.vin,
        source.acquisition_date,
        source.status,
        source.current_location,
        source.ingestion_timestamp,
        source.source_file
    )
""")

display(result)

print("Trailer Bronze MERGE completed successfully.")

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76e4064a-6d8b-48dc-9ab5-91cff239b9dc)

Trailer Bronze MERGE completed successfully.


In [16]:
bronze_count = spark.sql("""
SELECT COUNT(*) AS count
FROM bronze_trailers
""").collect()[0]["count"]

print(f"Bronze trailer records: {bronze_count}")

display(
    spark.sql("""
    SELECT *
    FROM bronze_trailers
    LIMIT 10
    """)
)

StatementMeta(, 7fb64ed4-7af4-4bce-98c5-05ecbab7db58, 18, Finished, Available, Finished, False)

Bronze trailer records: 180


SynapseWidget(Synapse.DataFrame, 18a9e046-339c-4bc6-928b-36f69aadc1a5)